# Fintech Mobile App — Onboarding Funnel Analysis
## Phase 3: Funnel Analysis

The goal here is to measure conversion and drop-off at each stage, then break it down by channel, device, city tier, and age group to understand where users are falling off and which segments are struggling most.


## 1. Setup


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"]   = (12, 5)
plt.rcParams["axes.spines.top"]  = False
plt.rcParams["axes.spines.right"]= False

FUNNEL_ORDER = [
    "app_install", "app_open", "signup_started", "signup_completed",
    "kyc_initiated", "kyc_completed", "first_transaction",
    "notification_opted_in", "day1_return", "day7_return"
]

FUNNEL_LABELS = [
    "App Install", "App Open", "Signup Started", "Signup Completed",
    "KYC Initiated", "KYC Completed", "First Transaction",
    "Notification Opt-in", "Day-1 Return", "Day-7 Return"
]

LABEL_MAP = dict(zip(FUNNEL_ORDER, FUNNEL_LABELS))

print("✅ Libraries loaded")

In [ ]:
users  = pd.read_csv("../data/users_clean.csv",  parse_dates=["signup_date"])
events = pd.read_csv("../data/events_clean.csv", parse_dates=["event_ts"])

print(f"users  : {users.shape[0]:,} rows")
print(f"events : {events.shape[0]:,} rows")
display(users.head(3))

## 2. Overall Funnel — Conversion & Drop-off

For each stage: users reached, % of total installs, step-over-step conversion, and drop-off rate.


In [ ]:
# Count unique users per stage
stage_reach = (
    events.groupby("event_name")["user_id"]
    .nunique()
    .reindex(FUNNEL_ORDER)
    .fillna(0)
    .astype(int)
    .reset_index()
    .rename(columns={"event_name": "stage", "user_id": "users_reached"})
)

total = stage_reach.loc[stage_reach["stage"] == "app_install", "users_reached"].values[0]

# Step conversion & drop-off
step_conv, step_drop, pct_total = [], [], []
for i, row in stage_reach.iterrows():
    pct_total.append(round(row["users_reached"] / total * 100, 1))
    if i == 0:
        step_conv.append(100.0)
        step_drop.append(0.0)
    else:
        prev = stage_reach.loc[i-1, "users_reached"]
        curr = row["users_reached"]
        step_conv.append(round(curr / prev * 100, 1) if prev > 0 else 0.0)
        step_drop.append(round((prev - curr) / prev * 100, 1) if prev > 0 else 0.0)

stage_reach["pct_of_total"]    = pct_total
stage_reach["step_conversion"] = step_conv
stage_reach["step_dropoff"]    = step_drop
stage_reach["label"]           = stage_reach["stage"].map(LABEL_MAP)

# Pretty display
display_df = stage_reach[["label","users_reached","pct_of_total","step_conversion","step_dropoff"]].copy()
display_df.columns = ["Stage","Users Reached","% of Total","Step Conv %","Step Drop %"]
display(display_df.style
    .background_gradient(subset=["Step Drop %"], cmap="RdYlGn_r")
    .background_gradient(subset=["% of Total"],  cmap="Blues")
    .format({"% of Total": "{:.1f}%", "Step Conv %": "{:.1f}%", "Step Drop %": "{:.1f}%",
             "Users Reached": "{:,}"})
)

## 3. Funnel Chart

Interactive Plotly chart — hover for exact numbers.


In [ ]:
fig = go.Figure(go.Funnel(
    y       = stage_reach["label"].tolist(),
    x       = stage_reach["users_reached"].tolist(),
    textinfo= "value+percent previous",
    textposition = "inside",
    marker  = dict(
        color = [
            "#2C3E80","#3D52A0","#4F66B8","#637AC8",
            "#E06B3E","#D4522A","#C83818",
            "#5BA85A","#4A944A","#38803A"
        ]
    ),
    connector=dict(line=dict(color="rgba(0,0,0,0.1)", width=1)),
))

fig.update_layout(
    title=dict(text="Fintech App Onboarding Funnel — All Users (n=10,000)",
               font=dict(size=16)),
    margin=dict(l=20, r=20, t=60, b=20),
    height=520,
    paper_bgcolor="white",
    plot_bgcolor ="white",
)

fig.write_html("../data/funnel_overall.html")
fig.show()
print("✅ Saved to data/funnel_overall.html")

## 4. By Acquisition Channel

Referral users should convert better since they're coming in with some social proof already. Paid Ad volume is high but quality might be lower.


In [ ]:
channels = users["acquisition_channel"].unique()

channel_funnel = []
for ch in channels:
    ch_users = set(users[users["acquisition_channel"] == ch]["user_id"])
    ch_events = events[events["user_id"].isin(ch_users)]
    n_total   = len(ch_users)
    for stage in FUNNEL_ORDER:
        n = ch_events[ch_events["event_name"] == stage]["user_id"].nunique()
        channel_funnel.append({
            "channel": ch, "stage": stage,
            "users": n, "pct": round(n / n_total * 100, 1)
        })

ch_df = pd.DataFrame(channel_funnel)
ch_pivot = ch_df.pivot(index="stage", columns="channel", values="pct").reindex(FUNNEL_ORDER)
ch_pivot.index = FUNNEL_LABELS

print("Conversion rate (% of channel users) at each stage:")
display(ch_pivot.style
    .background_gradient(cmap="Blues", axis=1)
    .format("{:.1f}%")
)

In [ ]:
ch_conv = (
    users.groupby("acquisition_channel")["converted"]
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
    .reset_index()
)
ch_conv.columns = ["Channel", "Conversion Rate (%)"]

colors = {"Referral":"#2C7BB6","Organic":"#4DAC26","Social Media":"#F4A900","Paid Ad":"#D7191C"}

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(ch_conv["Channel"], ch_conv["Conversion Rate (%)"],
               color=[colors.get(c,"#888") for c in ch_conv["Channel"]],
               edgecolor="white", height=0.55)

for bar, val in zip(bars, ch_conv["Conversion Rate (%)"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=11, fontweight="bold")

ax.set_xlabel("Conversion Rate (%)", fontsize=11)
ax.set_title("Final Conversion Rate by Acquisition Channel", fontsize=13, fontweight="bold", pad=12)
ax.set_xlim(0, ch_conv["Conversion Rate (%)"].max() * 1.2)
ax.axvline(users["converted"].mean()*100, color="gray", linestyle="--", alpha=0.6, linewidth=1)
ax.text(users["converted"].mean()*100 + 0.2, -0.5, "Avg", color="gray", fontsize=9)

plt.tight_layout()
plt.savefig("../data/conversion_by_channel.png", dpi=150, bbox_inches="tight")
plt.show()
print(ch_conv.to_string(index=False))

In [ ]:
# KYC drop-off broken down by channel
kyc_drop = []
for ch in channels:
    ch_users  = set(users[users["acquisition_channel"] == ch]["user_id"])
    ch_events = events[events["user_id"].isin(ch_users)]
    n_kyc_init = ch_events[ch_events["event_name"]=="kyc_initiated"]["user_id"].nunique()
    n_kyc_done = ch_events[ch_events["event_name"]=="kyc_completed"]["user_id"].nunique()
    drop = round((n_kyc_init - n_kyc_done) / n_kyc_init * 100, 1) if n_kyc_init > 0 else 0
    kyc_drop.append({"Channel": ch, "KYC Initiated": n_kyc_init,
                     "KYC Completed": n_kyc_done, "KYC Drop-off %": drop})

kyc_df = pd.DataFrame(kyc_drop).sort_values("KYC Drop-off %", ascending=False)
print("KYC drop-off by acquisition channel:")
display(kyc_df.style
    .background_gradient(subset=["KYC Drop-off %"], cmap="RdYlGn_r")
    .format({"KYC Drop-off %": "{:.1f}%"})
)

## 5. By Device Type

Not expecting a huge difference here, but worth checking — iOS vs Android can sometimes vary by income demographic.


In [ ]:
device_conv = (
    users.groupby("device_type")["converted"]
    .agg(["sum","count","mean"])
    .rename(columns={"sum":"converted_users","count":"total_users","mean":"conv_rate"})
    .assign(conv_rate=lambda x: (x["conv_rate"]*100).round(1))
    .reset_index()
)
print("Conversion by device type:")
display(device_conv)

# Stage-by-stage comparison
fig, ax = plt.subplots(figsize=(12, 5))
device_data = {}
for device in users["device_type"].unique():
    d_users  = set(users[users["device_type"] == device]["user_id"])
    d_events = events[events["user_id"].isin(d_users)]
    n_total  = len(d_users)
    pcts = [d_events[d_events["event_name"]==s]["user_id"].nunique()/n_total*100
            for s in FUNNEL_ORDER]
    device_data[device] = pcts

x = np.arange(len(FUNNEL_LABELS))
width = 0.35
colors_d = {"Android":"#4C72B0","iOS":"#DD8452"}

for i, (device, pcts) in enumerate(device_data.items()):
    ax.bar(x + i*width, pcts, width, label=device,
           color=colors_d.get(device,"#888"), edgecolor="white", alpha=0.9)

ax.set_xticks(x + width/2)
ax.set_xticklabels(FUNNEL_LABELS, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("% of Users Reaching Stage")
ax.set_title("Funnel Stage Reach by Device Type", fontsize=13, fontweight="bold", pad=12)
ax.legend()
plt.tight_layout()
plt.savefig("../data/funnel_by_device.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. By City Tier

Tier 1 cities might have better connectivity and more digital payment familiarity, but Tier 2/3 are catching up fast so the gap might not be as big as expected.


In [ ]:
tier_conv = (
    users.groupby("city_tier")["converted"]
    .agg(["sum","count","mean"])
    .rename(columns={"sum":"converted","count":"total","mean":"rate"})
    .assign(rate=lambda x: (x["rate"]*100).round(1))
)
print("Conversion by city tier:")
display(tier_conv)

# Stage reach heatmap by tier
tier_stage = []
for tier in ["Tier 1","Tier 2","Tier 3"]:
    t_users  = set(users[users["city_tier"]==tier]["user_id"])
    t_events = events[events["user_id"].isin(t_users)]
    n_total  = len(t_users)
    for stage, label in zip(FUNNEL_ORDER, FUNNEL_LABELS):
        n = t_events[t_events["event_name"]==stage]["user_id"].nunique()
        tier_stage.append({"City Tier":tier,"Stage":label,"Reach %":round(n/n_total*100,1)})

ts_df = pd.DataFrame(tier_stage).pivot(index="City Tier",columns="Stage",values="Reach %")
ts_df = ts_df[FUNNEL_LABELS]  # preserve funnel order

fig, ax = plt.subplots(figsize=(14, 3))
sns.heatmap(ts_df, annot=True, fmt=".1f", cmap="Blues",
            linewidths=0.5, ax=ax, cbar_kws={"label":"% Reach"})
ax.set_title("Funnel Stage Reach % by City Tier", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right", fontsize=9)
plt.tight_layout()
plt.savefig("../data/funnel_by_tier.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. By Age Group

The 25–34 group tends to be the core fintech demographic so I'd expect them to convert better. Let's see if the data backs that up.


In [ ]:
# Create age buckets
users["age_group"] = pd.cut(
    users["age"],
    bins=[17,24,34,44,55],
    labels=["18–24","25–34","35–44","45–55"]
)

age_conv = (
    users.groupby("age_group", observed=True)["converted"]
    .agg(["sum","count","mean"])
    .rename(columns={"sum":"converted","count":"total","mean":"rate"})
    .assign(rate=lambda x: (x["rate"]*100).round(1))
    .reset_index()
)
print("Conversion by age group:")
display(age_conv)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Conversion rate
axes[0].bar(age_conv["age_group"].astype(str), age_conv["rate"],
            color=["#2C3E80","#3D52A0","#637AC8","#94A8D8"], edgecolor="white")
axes[0].set_title("Conversion Rate by Age Group", fontweight="bold")
axes[0].set_ylabel("Conversion Rate (%)")
for i, (_, row) in enumerate(age_conv.iterrows()):
    axes[0].text(i, row["rate"]+0.3, f"{row['rate']}%", ha="center", fontsize=10, fontweight="bold")

# User volume
axes[1].bar(age_conv["age_group"].astype(str), age_conv["total"],
            color=["#2C3E80","#3D52A0","#637AC8","#94A8D8"], edgecolor="white")
axes[1].set_title("Total Users by Age Group", fontweight="bold")
axes[1].set_ylabel("Users")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{int(x):,}"))

plt.tight_layout()
plt.savefig("../data/funnel_by_age.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. KYC Drop-off

KYC initiated → KYC completed is showing the biggest single-step loss. Looking at where this drop-off varies across channels and age groups.


In [ ]:
# Users who reached KYC initiated
kyc_users = set(events[events["event_name"]=="kyc_initiated"]["user_id"])
kyc_done  = set(events[events["event_name"]=="kyc_completed"]["user_id"])
kyc_abandoned = kyc_users - kyc_done

kyc_profile = users[users["user_id"].isin(kyc_users)].copy()
kyc_profile["kyc_completed"] = kyc_profile["user_id"].isin(kyc_done)

print(f"Users who reached KYC Initiated : {len(kyc_users):,}")
print(f"Users who completed KYC         : {len(kyc_done):,}")
print(f"Users who abandoned at KYC      : {len(kyc_abandoned):,} ({len(kyc_abandoned)/len(kyc_users)*100:.1f}%)")

# KYC completion by channel
print("\nKYC completion rate by channel:")
display(
    kyc_profile.groupby("acquisition_channel")["kyc_completed"]
    .agg(["sum","count","mean"])
    .rename(columns={"sum":"completed","count":"reached","mean":"rate"})
    .assign(rate=lambda x: (x["rate"]*100).round(1))
    .sort_values("rate", ascending=False)
)

In [ ]:
# KYC completion by device
print("KYC completion rate by device:")
display(
    kyc_profile.groupby("device_type")["kyc_completed"]
    .agg(["sum","count","mean"])
    .rename(columns={"sum":"completed","count":"reached","mean":"rate"})
    .assign(rate=lambda x: (x["rate"]*100).round(1))
)

# KYC completion by age group
kyc_profile["age_group"] = pd.cut(kyc_profile["age"],
    bins=[17,24,34,44,55], labels=["18–24","25–34","35–44","45–55"])

print("\nKYC completion rate by age group:")
display(
    kyc_profile.groupby("age_group", observed=True)["kyc_completed"]
    .agg(["sum","count","mean"])
    .rename(columns={"sum":"completed","count":"reached","mean":"rate"})
    .assign(rate=lambda x: (x["rate"]*100).round(1))
)

In [ ]:
kyc_heat = (
    kyc_profile.groupby(["acquisition_channel","age_group"], observed=True)["kyc_completed"]
    .mean()
    .mul(100)
    .round(1)
    .unstack("age_group")
)

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(kyc_heat, annot=True, fmt=".1f", cmap="RdYlGn",
            vmin=50, vmax=75, linewidths=0.5, ax=ax,
            cbar_kws={"label":"KYC Completion %"})
ax.set_title("KYC Completion Rate — Channel × Age Group", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Age Group")
ax.set_ylabel("Acquisition Channel")
plt.tight_layout()
plt.savefig("../data/kyc_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Key Findings

---

### Finding 1 — Big drop-off before users even see the product
Only **15.9% of installs** end up completing a first transaction. The worst part is that a big chunk is lost at App Install → App Open (−24.7%) and App Open → Signup Started (−28%) — before anyone's actually used anything.

**What to do:** Personalised push notifications in the first 24 hours post-install. Maybe also reconsider making users sign up before they can see what the app actually does.

---

### Finding 2 — KYC is the biggest single-step blocker
**35.8% of users** who start KYC never finish it. That's consistent with the Deloitte benchmark (38%), but it doesn't make it acceptable.

**What to do:** Break KYC into smaller steps and show a progress indicator. Real-time document validation would help — right now users might be failing silently.

---

### Finding 3 — Referral converts 3.6× better than Paid Ads
Referral: **25.1%** conversion. Paid Ad: **7.0%** — and Paid Ad is the biggest channel (34.7% of users).

**What to do:** Pull back paid ad spend and redirect into a referral rewards program. Even a small incentive (cashback, fee waiver) shifts the mix toward a channel that's massively more efficient.

---

### Finding 4 — City tier doesn't really matter
Tier 1, 2 and 3 all convert at roughly **15.8%**. Digital payment comfort seems pretty consistent across urban India now.

**What to do:** Don't under-invest in Tier 2/3 — same conversion rates, probably lower CPAs on paid channels.

---

### Finding 5 — Day-7 retention drop is steep
**43.8%** of Day-1 returning users are gone by Day 7.

**What to do:** Some kind of Day-2 or Day-4 re-engagement push based on what feature they haven't tried yet. Streak rewards or early transaction incentives might help anchor the habit.


In [ ]:
# Summary table of all segment conversion rates
print("=" * 50)
print("CONVERSION RATE SUMMARY")
print("=" * 50)

overall = users["converted"].mean() * 100
print(f"\nOverall : {overall:.1f}%\n")

for col, label in [("acquisition_channel","By Channel"),
                   ("device_type","By Device"),
                   ("city_tier","By City Tier")]:
    print(f"--- {label} ---")
    summary = (users.groupby(col)["converted"].mean() * 100).round(1).sort_values(ascending=False)
    for k, v in summary.items():
        diff = v - overall
        arrow = "▲" if diff > 0 else "▼"
        print(f"  {k:<20} {v:.1f}%  {arrow} {abs(diff):.1f}pp vs avg")
    print()

---
## Phase 3 done

Charts saved to `data/`:
- `funnel_overall.html`
- `conversion_by_channel.png`
- `funnel_by_device.png`
- `funnel_by_tier.png`
- `funnel_by_age.png`
- `kyc_heatmap.png`

Next: Phase 4 — Cohort & Retention Analysis
